In [ ]:
!pip install import-ipynb
!pip install torchmetrics

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

In [ ]:
cd "drive/MyDrive/University/Projects/KROK/DL Project/Code"

In [ ]:
ls

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import librosa
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [ ]:
class AudioBreakDataset(Dataset):
    def __init__(self, audio_dir, label_dir, chunk_ms=20):
        """
        Args:
            audio_dir (str): Directory containing audio files
            label_dir (str): Directory containing label CSV files
            chunk_ms (int): Size of chunks in milliseconds
        """
        self.audio_dir = Path(audio_dir)
        self.label_dir = Path(label_dir)
        self.chunk_ms = chunk_ms

        # Get all audio files and their corresponding labels
        self.audio_files = sorted([f for f in self.audio_dir.glob("*.mp3")])
        self.label_files = sorted([f for f in self.label_dir.glob("*.csv")])

        # Validate matching files
        assert len(self.audio_files) == len(self.label_files), \
            "Number of audio files and label files must match"

        # Pre-compute lengths of each file
        self.file_lengths = []
        self.cumulative_lengths = [0]

        for audio_file, label_file in zip(self.audio_files, self.label_files):
            # Process each file to get number of chunks
            spec, sr = self._audio_to_spectrogram(str(audio_file))
            chunks = self._create_chunks(spec, sr)
            labels = self._process_labels(str(label_file))
            length = min(len(chunks), len(labels))

            self.file_lengths.append(length)
            self.cumulative_lengths.append(
                self.cumulative_lengths[-1] + length
            )

    def _audio_to_spectrogram(self, audio_path):
        y, sr = librosa.load(audio_path)
        spectrogram = librosa.feature.melspectrogram(y=y, sr=sr)
        spectrogram_db = librosa.power_to_db(spectrogram)
        return spectrogram_db, sr

    def _create_chunks(self, spectrogram, sr):
        frames_per_chunk = int((self.chunk_ms/1000) * sr)
        chunks = []

        for i in range(0, spectrogram.shape[1], frames_per_chunk):
            chunk = spectrogram[:, i:i+frames_per_chunk]
            if chunk.shape[1] < frames_per_chunk:
                chunk = np.pad(
                    chunk,
                    ((0,0), (0, frames_per_chunk-chunk.shape[1]))
                )
            chunks.append(chunk)
        return np.array(chunks)

    def _process_labels(self, label_path):
        labels = pd.read_csv(label_path)
        chunk_labels = []

        for i in range(0, max(labels['end']), self.chunk_ms):
            chunk_start = i
            chunk_end = i + self.chunk_ms
            is_break = labels[
                ((labels['start'] <= chunk_start) &
                 (labels['end'] >= chunk_start)) |
                ((labels['start'] <= chunk_end) &
                 (labels['end'] >= chunk_end))
            ]['break'].any()
            chunk_labels.append(is_break)

        return np.array(chunk_labels)

    def __len__(self):
        return self.cumulative_lengths[-1]

    def __getitem__(self, idx):
        # Find which file this index belongs to
        file_idx = np.searchsorted(self.cumulative_lengths[1:], idx, side='right')
        local_idx = idx - self.cumulative_lengths[file_idx]

        # Load and process audio file
        spec, sr = self._audio_to_spectrogram(str(self.audio_files[file_idx]))
        chunks = self._create_chunks(spec, sr)

        # Load and process label file
        labels = self._process_labels(str(self.label_files[file_idx]))

        # Get specific chunk and label
        chunk = chunks[local_idx]
        label = labels[local_idx]

        return torch.FloatTensor(chunk), torch.FloatTensor([label])

'\naudio_dir = "path/to/audio/files"\nlabel_dir = "path/to/label/files"\n\ndataloader = create_dataloaders(\n    audio_dir=audio_dir,\n    label_dir=label_dir,\n    batch_size=32\n)\n\n# In your training loop:\nfor batch_chunks, batch_labels in dataloader:\n    # batch_chunks.shape: [batch_size, n_mel_frequencies, time_steps]\n    # batch_labels.shape: [batch_size, 1]\n    # Feed to your RNN model\n    ...\n'

In [ ]:
# Usage example:
def create_dataloaders(audio_dir, label_dir, batch_size=32, num_workers=4):
    dataset = AudioBreakDataset(audio_dir, label_dir)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    return dataloader

In [ ]:
audio_dir = "path/to/audio/files"
label_dir = "path/to/label/files"

dataloader = create_dataloaders(
    audio_dir=audio_dir,
    label_dir=label_dir,
    batch_size=32
)

# In your training loop:
for batch_chunks, batch_labels in dataloader:
    # batch_chunks.shape: [batch_size, n_mel_frequencies, time_steps]
    # batch_labels.shape: [batch_size, 1]
    # Feed to your RNN model
